In [2]:
import pandas as pd
from ydata_profiling import ProfileReport

c:\Users\user\OneDrive\Documentos\Mineracao_dados_absenteismo\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\user\AppData\Local\Temp\ipykernel_10412\2829373970.py:2: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


In [3]:
df = pd.read_csv("Absenteeism_at_work_v2.csv", delimiter=";")


# Aplicando módulo para idades e valores de Distancia Negativas negativas 

In [4]:
df.loc[:, 'Age'] = df['Age'].abs()
df.loc[:, 'Distance from Residence to Work'] = df['Distance from Residence to Work'].abs()


# Retirando linhas com idades inconsistentes

# Removendo linha cujo mês estão fora do escopo de 1 à 12

In [5]:
df = df[df['Month of absence'].between(1,12)]
df = df[df['Age'] < 100]
df = df[df['Age'] >=18]

# Trantando colunas Vazias, substituindo por suas medianas

In [6]:
df['Distance from Residence to Work'] = df['Distance from Residence to Work'].fillna(df['Distance from Residence to Work'].median())
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Transportation expense'] = df['Transportation expense'].fillna(df['Transportation expense'].median())
df['Service time'] = df['Service time'].fillna(df['Service time'].median())
df['Work load Average/day '] = df['Work load Average/day '].fillna(df['Work load Average/day '].median())
df['Hit target'] = df['Hit target'].fillna(df['Hit target'].median())
df['Son'] = df['Son'].fillna(df['Son'].median())
df['Pet'] = df['Pet'].fillna(df['Pet'].median())
df['Absenteeism time in hours'] = df['Absenteeism time in hours'].fillna(df['Absenteeism time in hours'].median())
df['Body mass index'] = df['Body mass index'].fillna(df['Body mass index'].median())


# Retirando linhas vazias de dados Categoricos

In [7]:
df.dropna(subset=['Reason for absence', 'Day of the week', 'Seasons', 
    'Disciplinary failure','Social drinker','Social smoker'], inplace=True)

# Retirando colunas que não são relacionadas ao problema

In [8]:
df = df.drop(columns=['ID','Education','Weight','Height'])

# Criação da Variável Categórica (Risk_Class) e Divisão de Treino/Teste

In [9]:
import numpy as np

In [10]:
cortes = [-np.inf, 4, 8, np.inf]
class_names = ['0', '1', '2']
df['Risk_Class'] = pd.cut(df['Absenteeism time in hours'], 
                          bins=cortes, labels=class_names)

In [11]:
print("--- Distribuição das Classes de Risco ---")
print(df['Risk_Class'].value_counts())

--- Distribuição das Classes de Risco ---
Risk_Class
0    8139
1    3335
2    1992
Name: count, dtype: int64


# Salvando dataset limpo

In [12]:
df.to_csv('absenteismo_limpo.csv', index=False, encoding='utf-8')

# Gerando Gŕaficos para Análise Exploratoria

In [13]:
profile = ProfileReport(df)
profile.to_file("absenteismo_report.html")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 10.36it/s]


In [14]:
import matplotlib.pyplot as plt
import seaborn as sns

In [15]:
sns.set_theme(style="whitegrid")

In [16]:
plt.figure(figsize=(12, 8))

<Figure size 1200x800 with 0 Axes>

In [17]:
colunas_numericas = df.select_dtypes(include=['int64', 'float64']).columns
matriz_correlacao = df[colunas_numericas].corr()

In [18]:
sns.heatmap(matriz_correlacao, annot=False, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Matriz de Correlação entre Variáveis', fontsize=16)
plt.tight_layout()
plt.savefig('Matriz_de_Correlação.png')

In [19]:
plt.figure(figsize=(10, 6))
# Agrupa os dados pelo dia da semana e calcula a média de horas de falta
sns.barplot(data=df, x='Day of the week', y='Absenteeism time in hours', errorbar=None, palette='viridis')
plt.title('Média de Horas de Ausência por Dia da Semana', fontsize=16)
plt.xlabel('Dia da Semana (2=Segunda ... 6=Sexta)')
plt.ylabel('Média de Horas de Falta')
plt.tight_layout()
plt.savefig('Media_horas_falta_dia.png')

C:\Users\user\AppData\Local\Temp\ipykernel_10412\2843265510.py:3: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=df, x='Day of the week', y='Absenteeism time in hours', errorbar=None, palette='viridis')


# Média de Faltas por Número de filhos

In [20]:
plt.figure(figsize=(8, 6))
sns.barplot(data=df, x='Son', y='Absenteeism time in hours', errorbar=None, palette='mako')
plt.title('Impacto do Número de Filhos no Absenteísmo', fontsize=14)
plt.xlabel('Quantidade de Filhos')
plt.ylabel('Média de Horas de Ausência')
plt.tight_layout()

C:\Users\user\AppData\Local\Temp\ipykernel_10412\1790763673.py:2: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=df, x='Son', y='Absenteeism time in hours', errorbar=None, palette='mako')


In [21]:
plt.savefig('Grafico_Filhos.png')

In [22]:
plt.figure(figsize=(8, 6))
sns.barplot(data=df, x='Pet', y='Absenteeism time in hours', errorbar=None, palette='mako')
plt.title('Impacto do Número de Pets no Absenteísmo', fontsize=14)
plt.xlabel('Quantidade de Pets')
plt.ylabel('Média de Horas de Ausência')
plt.tight_layout()
plt.savefig('Grafico_Pets.png')

C:\Users\user\AppData\Local\Temp\ipykernel_10412\2583012888.py:2: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=df, x='Pet', y='Absenteeism time in hours', errorbar=None, palette='mako')


In [23]:
plt.figure(figsize=(10, 6))
sns.lineplot(data=df, x='Month of absence', y='Absenteeism time in hours', errorbar=None, color='crimson', marker='o', linewidth=2)
plt.title('Sazonalidade: Média de Horas de Ausência por Mês', fontsize=14)
plt.xlabel('Mês (1=Jan a 12=Dez)')
plt.ylabel('Média de Horas de Ausência')
plt.xticks(range(1, 13)) # Garante que todos os 12 meses apareçam no eixo X
plt.tight_layout()

In [24]:
plt.savefig('Grafico_Meses.png')

# Separando os Dados

In [25]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [26]:
colunas_para_remover = ['Absenteeism time in hours', 'Risk_Class']
X = df.drop(columns=colunas_para_remover)

# Alvos da Regressão e Classificação, respectivamente

In [27]:
y_reg = df['Absenteeism time in hours'] # Alvo da Regressão
y_clf = df['Risk_Class']                # Alvo da Classificação

# Divisão Treino e Teste (80% treino, 20% teste)

In [28]:
# Usamos o mesmo X para ambos, mas garantimos a proporção das classes com stratify
X_train, X_test,y_train_reg, y_test_reg = train_test_split(X, y_reg, test_size=0.2, random_state=42)
_, _, y_train_clf, y_test_clf = train_test_split(X, y_clf, test_size=0.2, random_state=42, stratify=y_clf)

# Padronização dos Dados 

In [29]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f"Dados separados e padronizados! Treino: {X_train.shape[0]} linhas | Teste: {X_test.shape[0]} linhas")

Dados separados e padronizados! Treino: 10772 linhas | Teste: 2694 linhas


# Treinando modelos Lineares

In [30]:
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, accuracy_score, classification_report

Regressão Linear

In [31]:
lin_reg = LinearRegression()
lin_reg.fit(X_train_scaled, y_train_reg)
y_pred_lin = lin_reg.predict(X_test_scaled)
rmse_lin = np.sqrt(mean_squared_error(y_test_reg, y_pred_lin))
print(f"Regressão Linear - Erro Médio (RMSE): {rmse_lin:.2f} horas")

Regressão Linear - Erro Médio (RMSE): 11.63 horas


Regressão Logistica

In [32]:
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train_clf)
y_pred_log = log_reg.predict(X_test_scaled)
acc_log = accuracy_score(y_test_clf, y_pred_log)
print(f"Regressão Logística - Acurácia: {acc_log*100:.2f}%\n")

Regressão Logística - Acurácia: 60.43%



# Treinando modelos de árvores de decisão

Árvore de Regressão

In [33]:
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier

In [45]:
MAX_DEPTH = 5

In [47]:
tree_reg = DecisionTreeRegressor()
tree_reg.fit(X_train, y_train_reg) 
y_pred_tree_reg = tree_reg.predict(X_test)
rmse_tree = np.sqrt(mean_squared_error(y_test_reg, y_pred_tree_reg))
print(f"Árvore de Regressão - Erro Médio (RMSE): {rmse_tree:.2f} horas")

Árvore de Regressão - Erro Médio (RMSE): 3.27 horas


In [60]:
tree_reg = DecisionTreeRegressor(random_state=42, criterion='squared_error', min_samples_leaf=1, min_samples_split=10)
tree_reg.fit(X_train, y_train_reg) 
y_pred_tree_reg = tree_reg.predict(X_test)
rmse_tree = np.sqrt(mean_squared_error(y_test_reg, y_pred_tree_reg))
print(f"Árvore de Regressão - Erro Médio (RMSE): {rmse_tree:.2f} horas")

Árvore de Regressão - Erro Médio (RMSE): 3.01 horas


Árvore de Classificação

In [81]:
tree_clf = DecisionTreeClassifier()
tree_clf.fit(X_train, y_train_clf)
y_pred_tree_clf = tree_clf.predict(X_test)
acc_tree = accuracy_score(y_test_clf, y_pred_tree_clf)

print(f"Árvore de Classificação - Acurácia: {acc_tree*100:.2f}%\n")

print("Relatório da Árvore de Classificação:")
print(classification_report(y_test_clf, y_pred_tree_clf))

Árvore de Classificação - Acurácia: 44.28%

Relatório da Árvore de Classificação:
              precision    recall  f1-score   support

           0       0.60      0.61      0.60      1628
           1       0.23      0.23      0.23       667
           2       0.13      0.13      0.13       399

    accuracy                           0.44      2694
   macro avg       0.32      0.32      0.32      2694
weighted avg       0.44      0.44      0.44      2694



In [ ]:
tree_clf = DecisionTreeClassifier(min_samples_leaf=1, min_samples_split=10, criterion="entropy", max_depth=3)
tree_clf.fit(X_train, y_train_clf)
y_pred_tree_clf = tree_clf.predict(X_test)
acc_tree = accuracy_score(y_test_clf, y_pred_tree_clf)

print(f"Árvore de Classificação - Acurácia: {acc_tree*100:.2f}%\n")

print("Relatório da Árvore de Classificação:")
print(classification_report(y_test_clf, y_pred_tree_clf))

Árvore de Classificação - Acurácia: 60.43%

Relatório da Árvore de Classificação:
              precision    recall  f1-score   support

           0       0.60      1.00      0.75      1628
           1       0.00      0.00      0.00       667
           2       0.33      0.00      0.00       399

    accuracy                           0.60      2694
   macro avg       0.31      0.33      0.25      2694
weighted avg       0.41      0.60      0.46      2694



c:\Users\user\OneDrive\Documentos\Mineracao_dados_absenteismo\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\user\OneDrive\Documentos\Mineracao_dados_absenteismo\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\user\OneDrive\Documentos\Mineracao_dados_absenteismo\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to

In [38]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier

In [39]:
parametros_arvore = {
    'criterion': ['gini', 'entropy'],      # Como a árvore toma decisões
    'max_depth': [3, 5, 7, 10, None],      # Profundidade (None = cresce até o fim, perigoso!)
    'min_samples_split': [2, 5, 10, 20],   # Mínimo para dividir um grupo
    'min_samples_leaf': [1, 2, 5, 10]      # Mínimo de pessoas na resposta final
}

In [40]:
tree_base = DecisionTreeClassifier(random_state=42)

In [41]:
grid_tree = GridSearchCV(estimator=tree_base, param_grid=parametros_arvore, cv=5, n_jobs=-1, scoring='accuracy')

In [42]:
grid_tree.fit(X_train, y_train_clf)

,estimator,DecisionTreeC...ndom_state=42)
,param_grid,"{'criterion': ['gini', 'entropy'], 'max_depth': [3, 5, ...], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 5, ...]}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,criterion,'entropy'


In [43]:
print(f"Melhor combinação de Hiperparâmetros encontrada:")
print(grid_tree.best_params_)

Melhor combinação de Hiperparâmetros encontrada:
{'criterion': 'entropy', 'max_depth': 3, 'min_samples_leaf': 1, 'min_samples_split': 10}


In [44]:
print(f"\n🎯 Acurácia (em Validação Cruzada) com a Árvore Otimizada: {grid_tree.best_score_*100:.2f}%")


🎯 Acurácia (em Validação Cruzada) com a Árvore Otimizada: 60.46%
